# Fridge Detector — Kaggle Training

## Before running, attach these 3 datasets to this notebook:

1. **Source code** — zip and upload your project:
   ```bash
   zip -r fridge_detector_src.zip src/ scripts/ configs/ pyproject.toml -x '*.pyc' -x '__pycache__/*'
   ```
   Upload on kaggle.com: Datasets → New Dataset → `fridge_detector_src.zip`
   → Kaggle mounts at: `/kaggle/input/datasets/magicayyub/fridge-detector/`

2. **Images** — use the existing public dataset (images only):
   https://www.kaggle.com/datasets/mayarmohamedswilam/freiburg-groceries
   → Kaggle mounts at: `/kaggle/input/datasets/mayarmohamedswilam/freiburg-groceries/images/`

3. **Annotations** — your uploaded dataset (XML files, 19 MB):
   https://www.kaggle.com/datasets/magicayyub/freiburg-groceries-annotations
   → Kaggle mounts at: `/kaggle/input/datasets/magicayyub/freiburg-groceries-annotations/annotations/`

The paths in cell 2 are already configured correctly for these mount points. Just attach all 3 datasets and run.

In [ ]:
# ── Configure these paths to match your Kaggle dataset mounts ──────────────
# Note: Kaggle mounts datasets at /kaggle/input/datasets/<owner>/<slug>/
SRC_DIR    = '/kaggle/input/datasets/magicayyub/fridge-detector'                 # your source code
IMAGES_DIR = '/kaggle/input/datasets/mayarmohamedswilam/freiburg-groceries/images'  # actual images subdir
ANNOTS_DIR = '/kaggle/input/datasets/magicayyub/freiburg-groceries-annotations/annotations'  # annotations
# ──────────────────────────────────────────────────────────────────────────────

import os, sys
sys.path.insert(0, f'{SRC_DIR}/src')
os.environ['KAGGLE_SRC']    = SRC_DIR
os.environ['KAGGLE_IMAGES'] = IMAGES_DIR
os.environ['KAGGLE_ANNOTS'] = ANNOTS_DIR

print('Python:', sys.version)
print('SRC_DIR exists:    ', os.path.isdir(SRC_DIR))
print('IMAGES_DIR exists: ', os.path.isdir(IMAGES_DIR))
print('ANNOTS_DIR exists: ', os.path.isdir(ANNOTS_DIR))

In [ ]:
# Install the one missing package (torch / torchvision already on Kaggle)
!pip install rich pyyaml -q

In [ ]:
# Verify GPU and imports
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

from utils import get_device
from data.dataset import VOCDetectionDataset

# Images are in BEANS/, CAKE/ etc. (uppercase) — the loader handles case-insensitive matching
ds = VOCDetectionDataset(
    IMAGES_DIR,
    f'{ANNOTS_DIR}/annotations',
    VOCDetectionDataset.FREIBURG_CLASSES,
    image_size=512,
)
print(f'Dataset: {len(ds)} samples, {len(VOCDetectionDataset.FREIBURG_CLASSES)} classes')

In [ ]:
# Debug: trace path matching
import os

print("=== Tracing VOCDetectionDataset path matching ===\n")

# Simulate the image index building
img_index = {}
for dirpath, _, filenames in os.walk(IMAGES_DIR):
    for fname in filenames:
        if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue
        full = os.path.join(dirpath, fname)
        rel = os.path.relpath(full, IMAGES_DIR).lower()
        img_index[rel] = full

print(f"Image index has {len(img_index)} entries")
print(f"Sample keys (first 3): {list(img_index.keys())[:3]}\n")

# Simulate annotation matching
matched = 0
for dirpath, _, filenames in os.walk(ANNOTS_DIR):
    rel_dir = os.path.relpath(dirpath, ANNOTS_DIR)
    for fname in sorted(filenames):
        if not fname.endswith('.xml'):
            continue
        stem = fname[:-4]
        key = os.path.join(rel_dir, stem + '.jpg').lower()
        if key in img_index:
            matched += 1
            if matched <= 2:  # Show first 2 matches
                print(f"✓ MATCH: {key} → {img_index[key]}")
        else:
            if matched == 0 and fname.endswith('.xml'):  # Show first mismatch
                print(f"✗ NO MATCH: {key}")
                print(f"  Looking for: {key}")
                print(f"  XML at: {os.path.join(dirpath, fname)}")

print(f"\nMatched {matched} samples out of {len(img_index)} images")
print(f"\n=== ANNOTS_DIR contents ===")
print(f"Path: {ANNOTS_DIR}")
for i, item in enumerate(sorted(os.listdir(ANNOTS_DIR))[:5]):
    item_path = os.path.join(ANNOTS_DIR, item)
    if os.path.isdir(item_path):
        xmls = [f for f in os.listdir(item_path) if f.endswith('.xml')]
        print(f"  {item}/ → {len(xmls)} XMLs (sample: {xmls[0] if xmls else 'none'})")

In [ ]:
# ── Train ─────────────────────────────────────────────────────────────────────
# kaggle.yaml sets batch_size=16, epochs=50, num_workers=4.
# Override any value by appending CLI args, e.g. --epochs 30

import subprocess
import os

env = os.environ.copy()
env['PYTHONPATH'] = f'{SRC_DIR}/src'

result = subprocess.run([
    'python', f'{SRC_DIR}/scripts/train.py',
    '--config', f'{SRC_DIR}/configs/kaggle.yaml',
    '--data-dir', IMAGES_DIR,
    '--annot-dir', ANNOTS_DIR  # Already includes /annotations
], env=env)

exit(result.returncode)

In [ ]:
# List saved checkpoints
import glob
for ckpt in sorted(glob.glob('/kaggle/working/checkpoints/*.pt')):
    size_mb = os.path.getsize(ckpt) / 1e6
    print(f'{ckpt}  ({size_mb:.1f} MB)')